# Tree Ensembles: Forests and Boosting

This notebook compares a single tree, random forests, extremely randomized trees, gradient boosting, and histogram gradient boosting on the same nonlinear classification task.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 11

In [ ]:
X, y = make_moons(n_samples=2500, noise=0.28, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=RANDOM_STATE
)

plt.figure(figsize=(6, 4.5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=12, cmap="coolwarm", alpha=0.65)
plt.title("Synthetic nonlinear tabular task")
plt.xlabel("x0")
plt.ylabel("x1")
plt.show()

In [ ]:
models = {
    "CART depth=4": DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, random_state=RANDOM_STATE),
}

rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
        "fit_seconds": elapsed,
    })

scores = pd.DataFrame(rows).sort_values("roc_auc", ascending=False)
scores

In [ ]:
def plot_boundary(model, ax, title):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 250), np.linspace(y_min, y_max, 250))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, zz, levels=np.linspace(0, 1, 13), cmap="coolwarm", alpha=0.70)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, s=8, cmap="coolwarm", edgecolor="none", alpha=0.55)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, model) in zip(axes.ravel(), models.items()):
    plot_boundary(model, ax, name)
axes.ravel()[-1].axis("off")
fig.suptitle("Decision boundaries: variance reduction vs additive correction")
fig.tight_layout()

In [ ]:
best_name = scores.iloc[0]["model"]
best_model = models[best_name]
perm = permutation_importance(best_model, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)
pd.DataFrame({
    "feature": ["x0", "x1"],
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

## Takeaway

Random forests reduce variance by averaging many high-variance trees. Boosting builds an additive model that corrects residual mistakes. Both are ensembles of trees, but their error-reduction logic is different.